# Day 185 — AWS S3 Part 1: Buckets, Upload, Download & boto3
**Month 11 | Data Engineering Portfolio | LogiTrack India**

---
| Field | Detail |
|---|---|
| **Topic** | AWS S3 fundamentals via boto3 (mocked with moto — no AWS account needed) |
| **Dataset** | LogiTrack India · seed=181 · 538 raw rows · 530 clean rows |
| **Points** | 90 base + 10★ bonus |
| **Structure** | Section 1: Raw Data · Section 2: Concept Notes · Section 3: Tasks · Section 4: Answer Key · Section 5: Rubric |

> **Why this matters:** S3 is the most common cloud storage layer in production data pipelines. Programmatically pushing/pulling files from S3 with boto3 is a baseline expectation for any data engineering freelance project touching AWS.


## ⚙️ Environment Setup

In [ ]:
# Goal: Install boto3 (AWS SDK for Python) and moto (S3 mock library)
# Method: pip install — run this cell once, then Runtime → Restart Runtime
!pip install boto3 moto --quiet


In [ ]:
# Goal: Import libraries; set fake AWS credentials so moto works without a real AWS account
# Method: os.environ — moto reads these values but never contacts real AWS

import boto3, io, os
import numpy as np
import pandas as pd
from moto import mock_aws

os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'   # Mumbai
os.environ['AWS_ACCESS_KEY_ID']     = 'testing'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'testing'
os.environ['AWS_SECURITY_TOKEN']    = 'testing'
os.environ['AWS_SESSION_TOKEN']     = 'testing'

print("boto3 version:", boto3.__version__)
print("Environment: moto mock configured")


---
## Section 1 — Raw Data
> ⚠️ **DO NOT MODIFY THIS CELL**

In [ ]:
# ── RAW DATA — DO NOT MODIFY ──────────────────────────────────────────
np.random.seed(181)
n = 530
regions    = ['North', 'South', 'East', 'West', 'Central']
categories = ['Electronics', 'Apparel', 'Food', 'Furniture', 'Pharma']
carriers   = ['BlueDart', 'DTDC', 'Delhivery', 'Ecom Express', 'XpressBees']
statuses   = ['Delivered', 'In Transit', 'Delayed', 'Returned']
status_w   = [0.60, 0.20, 0.15, 0.05]

data = {
    'shipment_id'    : [f'SHP{str(i).zfill(5)}' for i in range(1, n + 1)],
    'region'         : np.random.choice(regions, n),
    'category'       : np.random.choice(categories, n),
    'carrier'        : np.random.choice(carriers, n),
    'weight_kg'      : np.round(np.random.uniform(0.5, 50.0, n), 2),
    'shipping_cost'  : np.round(np.random.uniform(50, 2000, n), 2),
    'days_in_transit': np.random.randint(1, 21, n),
    'status'         : np.random.choice(statuses, n, p=status_w),
    'delivery_date'  : pd.date_range('2024-01-01', periods=n, freq='12h').strftime('%Y-%m-%d'),
}
df_base = pd.DataFrame(data)
dup_rows = df_base.iloc[:8].copy()
df_raw   = pd.concat([df_base, dup_rows], ignore_index=True)

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)


---
## Section 2 — Concept Notes

### What is Amazon S3?
**S3 (Simple Storage Service)** stores files as **objects** inside **buckets**. Unlike a file system, S3 uses flat key-based addressing — "folders" are just key prefixes.

| Term | Meaning | Example |
|------|---------|---------|
| **Bucket** | Top-level container (globally unique) | `logitrack-india-bucket` |
| **Object** | A file stored in S3 | CSV, parquet, JSON |
| **Key** | Full "path" of the object | `logistic/raw/logitrack_raw.csv` |
| **Region** | AWS data center | `ap-south-1` (Mumbai) |
| **Prefix** | Key prefix — simulates a folder | `logistic/raw/` |

### boto3 Core Methods
```python
s3 = boto3.client('s3', region_name='ap-south-1')

s3.create_bucket(Bucket=..., CreateBucketConfiguration={...})
s3.put_object(Bucket=..., Key=..., Body=...)          # upload
s3.get_object(Bucket=..., Key=...)                     # download
s3.list_objects_v2(Bucket=..., Prefix=...)             # list
```

### Upload/Download Pattern (the fundamental primitive)
```python
# Upload: DataFrame → BytesIO → S3
buf = io.BytesIO()
df.to_csv(buf, index=False)
buf.seek(0)
s3.put_object(Bucket='my-bucket', Key='folder/file.csv', Body=buf.getvalue())

# Download: S3 → BytesIO → DataFrame
obj = s3.get_object(Bucket='my-bucket', Key='folder/file.csv')
df  = pd.read_csv(io.BytesIO(obj['Body'].read()))
```

### Why moto?
`moto` intercepts all boto3 calls and runs them against an in-memory fake AWS — **zero charges, zero real credentials**.

```python
from moto import mock_aws

@mock_aws
def my_function():
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(...)   # runs 100% locally
```

### Production Pipeline Pattern
```
raw data ──► S3 logistic/raw/ ──► ETL ──► S3 logistic/processed/ ──► analytics
             (landing zone)               (clean layer)
```

### Interview Answer
> *"I build ETL pipelines where S3 is the landing zone — raw files arrive in a raw/ prefix, a transform step cleans them, and results land in a processed/ prefix. boto3's `put_object` / `get_object` with a BytesIO buffer is the standard pattern. For unit tests, I use moto to mock S3 entirely so CI pipelines need no real AWS infrastructure."*


---
## Section 3 — Practice Tasks *(90 points + 10★ bonus)*

| Task | Topic | Points |
|------|-------|--------|
| T1 | Create S3 bucket + verify HTTP status | 15 |
| T2 | Upload raw CSV to S3 | 15 |
| T3 | List objects in bucket (count + size) | 20 |
| T4 | Download CSV from S3 + load into pandas | 20 |
| T5 | Transform + upload processed CSV | 10 |
| T6 | NRA insight from carrier transit data | 10 |
| ★ Bonus | Prefix-based listing (raw vs processed) | 10★ |


### Task 1 — Create S3 Bucket (15 pts)
Create bucket `logitrack-india-bucket` in `ap-south-1` using `boto3.client`. Print the bucket name and HTTP status code from `response['ResponseMetadata']['HTTPStatusCode']`.

In [ ]:
# Goal: Create an S3 bucket inside a moto mock context
# Method: @mock_aws decorator → boto3.client('s3') → create_bucket with LocationConstraint

@mock_aws
def task1_create_bucket():
    # Step 1: Create S3 client for ap-south-1
    s3 = ___

    # Step 2: Create bucket (non-us-east-1 requires CreateBucketConfiguration)
    response = s3.create_bucket(
        Bucket=___,
        CreateBucketConfiguration={'LocationConstraint': ___}
    )

    # Step 3: Print bucket name and HTTP status
    print(f"Bucket created : ___")
    print(f"HTTP status    : ___")
    return response

task1_create_bucket()


In [6]:
# Goal: Create an S3 bucket inside a moto mock context
# Method: @mock_aws decorator → boto3.client('s3') → create_bucket with LocationConstraint

@mock_aws
def task1_create_bucket():
    # Step 1: Create S3 client for ap-south-1
    s3 = boto3.client('s3', region_name='ap-south-1')

    # Step 2: Create bucket (non-us-east-1 requires CreateBucketConfiguration)
    response = s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )

    # Step 3: Print bucket name and HTTP status
    print(f"Bucket created : logitrack-india-bucket")
    print(f"HTTP status    : {response['ResponseMetadata']['HTTPStatusCode']}")
    return response

task1_create_bucket()

Bucket created : logitrack-india-bucket
HTTP status    : 200


{'ResponseMetadata': {'RequestId': 'OdBgR4I7t7BAubCbfjlPiEMBC79I20a01zBfstGg89ZIqSuspwGb',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'location': '/logitrack-india-bucket',
   'server': 'amazon.com',
   'date': 'Fri, 03 Jul 2026 14:34:45 GMT',
   'x-amzn-requestid': 'OdBgR4I7t7BAubCbfjlPiEMBC79I20a01zBfstGg89ZIqSuspwGb'},
  'RetryAttempts': 0},
 'Location': '/logitrack-india-bucket'}

### Task 2 — Upload Raw CSV to S3 (15 pts)
Inside `@mock_aws`: recreate the bucket, then upload `df_raw` as CSV to key `logistic/raw/logitrack_raw.csv` using `io.BytesIO`. Print upload HTTP status.

In [ ]:
# Goal: Serialize DataFrame to bytes, push to S3 with put_object
# Method: io.BytesIO buffer → df.to_csv → buf.seek(0) → s3.put_object

@mock_aws
def task2_upload_raw():
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )

    # Step 1: Write df_raw to an in-memory buffer
    buf = ___
    df_raw.to_csv(___, index=False)
    buf.seek(0)

    # Step 2: Upload to S3 with correct Bucket and Key
    response = s3.put_object(
        Bucket=___,
        Key=___,
        Body=___
    )

    # Step 3: Print upload HTTP status
    print(f"Upload status: ___")
    return response

task2_upload_raw()


In [7]:
# Goal: Serialize DataFrame to bytes, push to S3 with put_object
# Method: io.BytesIO buffer → df.to_csv → buf.seek(0) → s3.put_object

@mock_aws
def task2_upload_raw():
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )

    # Step 1: Write df_raw to an in-memory buffer
    buf = io.BytesIO()
    df_raw.to_csv(buf, index=False)
    buf.seek(0)

    # Step 2: Upload to S3 with correct Bucket and Key
    response = s3.put_object(
        Bucket='logitrack-india-bucket',
        Key='logistic/raw/logitrack_raw.csv',
        Body=buf.getvalue()
    )

    # Step 3: Print upload HTTP status
    print(f"Upload status: {response['ResponseMetadata']['HTTPStatusCode']}")
    return response

task2_upload_raw()

Upload status: 200


{'ResponseMetadata': {'RequestId': 'wqDAnp2kFjc7ZyDZuNO6NGo1LP3s0Wgp5lJpwrYg6IqLJBL9hO8g',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-checksum-crc32': 'Ci6Vjw==',
   'etag': '"4eb0b96e858383196035f34fbf04b06e"',
   'last-modified': 'Fri, 03 Jul 2026 14:34:58 GMT',
   'x-amz-sdk-checksum-algorithm': 'CRC32',
   'x-amzn-requestid': 'wqDAnp2kFjc7ZyDZuNO6NGo1LP3s0Wgp5lJpwrYg6IqLJBL9hO8g'},
  'RetryAttempts': 0},
 'ETag': '"4eb0b96e858383196035f34fbf04b06e"',
 'ChecksumCRC32': 'Ci6Vjw=='}

### Task 3 — List Objects in Bucket (20 pts)
Inside `@mock_aws`: recreate bucket + upload raw CSV (setup). Then call `list_objects_v2` with `Prefix='logistic/'`. Print: (a) total object count via `response['KeyCount']`, (b) each object's Key and Size.

In [ ]:
# Goal: Retrieve bucket inventory — what objects exist, how big
# Method: s3.list_objects_v2(Bucket, Prefix) → KeyCount + Contents[i]['Key'/'Size']

@mock_aws
def task3_list_objects():
    # Setup: create bucket + upload raw CSV
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())

    # Step 1: List all objects under logistic/ prefix
    response = s3.list_objects_v2(Bucket=___, Prefix=___)

    # Step 2: Print total count
    print(f"Total objects in 'logistic/' prefix: ___")

    # Step 3: Print each Key and Size
    for obj in response['___']:
        print(f"  Key: {obj['___']}  |  Size: {obj['___']} bytes")

    return response

task3_list_objects()


In [8]:
# Goal: Retrieve bucket inventory — what objects exist, how big
# Method: s3.list_objects_v2(Bucket, Prefix) → KeyCount + Contents[i]['Key'/'Size']

@mock_aws
def task3_list_objects():
    # Setup: create bucket + upload raw CSV
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())

    # Step 1: List all objects under logistic/ prefix
    response = s3.list_objects_v2(Bucket='logitrack-india-bucket', Prefix='logistic/')

    # Step 2: Print total count
    print(f"Total objects in 'logistic/' prefix: {response['KeyCount']}")

    # Step 3: Print each Key and Size
    for obj in response['Contents']:
        print(f"  Key: {obj['Key']}  |  Size: {obj['Size']} bytes")

    return response

task3_list_objects()

Total objects in 'logistic/' prefix: 1
  Key: logistic/raw/logitrack_raw.csv  |  Size: 38061 bytes


{'ResponseMetadata': {'RequestId': 'DCP463wvqteaRC8bvrNC9seJ55m43b9tx1vsV8kwtcBAXYYejZf2',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/xml',
   'server': 'amazon.com',
   'date': 'Fri, 03 Jul 2026 14:35:08 GMT',
   'x-amzn-requestid': 'DCP463wvqteaRC8bvrNC9seJ55m43b9tx1vsV8kwtcBAXYYejZf2'},
  'RetryAttempts': 0},
 'IsTruncated': False,
 'Contents': [{'Key': 'logistic/raw/logitrack_raw.csv',
   'LastModified': datetime.datetime(2026, 7, 3, 14, 35, 8, tzinfo=tzutc()),
   'ETag': '"4eb0b96e858383196035f34fbf04b06e"',
   'ChecksumAlgorithm': ['CRC32'],
   'Size': 38061,
   'StorageClass': 'STANDARD'}],
 'Name': 'logitrack-india-bucket',
 'Prefix': 'logistic/',
 'MaxKeys': 1000,
 'EncodingType': 'url',
 'KeyCount': 1}

### Task 4 — Download CSV from S3 + Load into Pandas (20 pts)
Inside `@mock_aws`: recreate bucket + upload. Use `get_object` to download `logistic/raw/logitrack_raw.csv`, then load into `df_downloaded` via `io.BytesIO`. Print shape and column names.

In [ ]:
# Goal: Pull bytes from S3 → reconstruct DataFrame in memory
# Method: s3.get_object → obj['Body'].read() → io.BytesIO → pd.read_csv

@mock_aws
def task4_download():
    # Setup
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())

    # Step 1: Download object from S3
    obj = s3.get_object(Bucket=___, Key=___)

    # Step 2: Read body bytes → DataFrame
    df_downloaded = pd.read_csv(io.BytesIO(obj['___'].read()))

    # Step 3: Print shape and columns
    print(f"Downloaded shape  : ___")
    print(f"Downloaded columns: ___")

    return df_downloaded

df_downloaded = task4_download()


In [9]:
# Goal: Pull bytes from S3 → reconstruct DataFrame in memory
# Method: s3.get_object → obj['Body'].read() → io.BytesIO → pd.read_csv

@mock_aws
def task4_download():
    # Setup
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())

    # Step 1: Download object from S3
    obj = s3.get_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv')

    # Step 2: Read body bytes → DataFrame
    df_downloaded = pd.read_csv(io.BytesIO(obj['Body'].read()))

    # Step 3: Print shape and columns
    print(f"Downloaded shape  : {df_downloaded.shape}")
    print(f"Downloaded columns: {list(df_downloaded.columns)}")

    return df_downloaded

df_downloaded = task4_download()

Downloaded shape  : (538, 9)
Downloaded columns: ['shipment_id', 'region', 'category', 'carrier', 'weight_kg', 'shipping_cost', 'days_in_transit', 'status', 'delivery_date']


### Task 5 — Transform + Upload Processed CSV (10 pts)
Using `df_downloaded`: (1) `drop_duplicates()` → `df_clean`, (2) add `delay_flag` column: `'High Delay'` if `days_in_transit > 14` else `'On Track'`, (3) upload `df_clean` to key `logistic/processed/logitrack_processed.csv`. Print clean row count, High Delay count, and upload HTTP status.

In [ ]:
# Goal: Apply ETL transforms in memory; write result to S3 processed/ layer
# Method: drop_duplicates → apply delay_flag → put_object

@mock_aws
def task5_transform_upload():
    # Setup: full pipeline
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())
    obj = s3.get_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv')
    df_dl = pd.read_csv(io.BytesIO(obj['Body'].read()))

    # Step 1: Drop duplicates
    df_clean = ___

    # Step 2: Add delay_flag
    df_clean['delay_flag'] = df_clean['days_in_transit'].apply(lambda x: ___)

    print(f"Clean rows     : ___")
    print(f"High Delay rows: ___")

    # Step 3: Upload processed CSV to logistic/processed/
    buf2 = io.BytesIO()
    df_clean.to_csv(buf2, index=False)
    buf2.seek(0)
    resp = s3.put_object(
        Bucket=___,
        Key=___,
        Body=buf2.getvalue()
    )
    print(f"Processed upload status: ___")
    return df_clean

df_clean = task5_transform_upload()


In [10]:
# Goal: Apply ETL transforms in memory; write result to S3 processed/ layer
# Method: drop_duplicates → apply delay_flag → put_object

@mock_aws
def task5_transform_upload():
    # Setup: full pipeline
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())
    obj = s3.get_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv')
    df_dl = pd.read_csv(io.BytesIO(obj['Body'].read()))

    # Step 1: Drop duplicates
    df_clean = df_dl.drop_duplicates().reset_index(drop=True)

    # Step 2: Add delay_flag
    df_clean['delay_flag'] = df_clean['days_in_transit'].apply(lambda x: 'High Delay' if x > 14 else 'On Track')

    print(f"Clean rows     : {len(df_clean)}")
    print(f"High Delay rows: {(df_clean['delay_flag'] == 'High Delay').sum()}")

    # Step 3: Upload processed CSV to logistic/processed/
    buf2 = io.BytesIO()
    df_clean.to_csv(buf2, index=False)
    buf2.seek(0)
    resp = s3.put_object(
        Bucket='logitrack-india-bucket',
        Key='logistic/processed/logitrack_processed.csv',
        Body=buf2.getvalue()
    )
    print(f"Processed upload status: {resp['ResponseMetadata']['HTTPStatusCode']}")
    return df_clean

df_clean = task5_transform_upload()

Clean rows     : 530
High Delay rows: 153
Processed upload status: 200


### Task 6 — NRA Business Insight (10 pts)
Compute carrier-level mean `days_in_transit` from `df_clean`. Write **one NRA bullet** identifying the worst-performing carrier.

Format:
```
• [Number] — [Reason] — [Action]
```

NRA Rules:
- **Number**: carrier name + exact avg days (2 dp) from printed output
- **Reason**: structural/causal mechanism — NOT the outcome or the benefit
- **Action**: specific, committed, names carrier + threshold + operational step. Zero hedging words.


In [ ]:
# Goal: Identify worst carrier by transit time, write NRA insight
# Method: groupby('carrier')['days_in_transit'].mean().round(2) → sort → NRA

carrier_avg = df_clean.groupby('carrier')['days_in_transit'].mean().round(2)
print(carrier_avg.sort_values(ascending=False))
print()

# ── YOUR NRA BULLET ──
nra = """
• [Number] — [Reason] — [Action]
"""
print(nra)


In [11]:
# Goal: Identify worst carrier by transit time, write NRA insight
# Method: groupby('carrier')['days_in_transit'].mean().round(2) → sort → NRA

carrier_avg = df_clean.groupby('carrier')['days_in_transit'].mean().round(2)
print(carrier_avg.sort_values(ascending=False))
print()

# ── YOUR NRA BULLET ──
nra = """
• XpressBees averages 11.42 days in transit — highest across all 5 carriers —
  because its network has fewer direct hub connections, adding relay stops per shipment.
  Reallocate ≥50% of XpressBees volume to Delhivery (9.73 days) for North and West
  zones; renegotiate XpressBees SLA to ≤10 days effective next quarter.
"""
print(nra)

carrier
XpressBees      11.42
Ecom Express    10.74
DTDC             9.89
BlueDart         9.86
Delhivery        9.73
Name: days_in_transit, dtype: float64


• XpressBees averages 11.42 days in transit — highest across all 5 carriers —
  because its network has fewer direct hub connections, adding relay stops per shipment.
  Reallocate ≥50% of XpressBees volume to Delhivery (9.73 days) for North and West
  zones; renegotiate XpressBees SLA to ≤10 days effective next quarter.



### ★ Bonus — Prefix-Based Listing (10★)
Inside `@mock_aws`, upload both raw and processed CSVs. Use `list_objects_v2` **twice** — once with `Prefix='logistic/raw/'`, once with `Prefix='logistic/processed/'`. Print object count per prefix.

Expected output:
```
Objects in logistic/raw/       : 1
Objects in logistic/processed/ : 1
```


In [ ]:
# Goal: Use prefix filter to list objects by "folder" — core S3 navigation skill
# Method: two list_objects_v2 calls with different Prefix values

@mock_aws
def bonus_prefix_listing():
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    # Upload raw
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())
    # Upload processed
    df_proc = df_raw.drop_duplicates().reset_index(drop=True)
    df_proc['delay_flag'] = df_proc['days_in_transit'].apply(lambda x: 'High Delay' if x > 14 else 'On Track')
    buf2 = io.BytesIO(); df_proc.to_csv(buf2, index=False); buf2.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/processed/logitrack_processed.csv', Body=buf2.getvalue())

    # YOUR CODE: list each prefix
    raw_list  = s3.list_objects_v2(Bucket=___, Prefix=___)
    proc_list = s3.list_objects_v2(Bucket=___, Prefix=___)

    print(f"Objects in logistic/raw/       : ___")
    print(f"Objects in logistic/processed/ : ___")

bonus_prefix_listing()


In [12]:
# Goal: Use prefix filter to list objects by "folder" — core S3 navigation skill
# Method: two list_objects_v2 calls with different Prefix values

@mock_aws
def bonus_prefix_listing():
    s3 = boto3.client('s3', region_name='ap-south-1')
    s3.create_bucket(
        Bucket='logitrack-india-bucket',
        CreateBucketConfiguration={'LocationConstraint': 'ap-south-1'}
    )
    # Upload raw
    buf = io.BytesIO(); df_raw.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/raw/logitrack_raw.csv', Body=buf.getvalue())
    # Upload processed
    df_proc = df_raw.drop_duplicates().reset_index(drop=True)
    df_proc['delay_flag'] = df_proc['days_in_transit'].apply(lambda x: 'High Delay' if x > 14 else 'On Track')
    buf2 = io.BytesIO(); df_proc.to_csv(buf2, index=False); buf2.seek(0)
    s3.put_object(Bucket='logitrack-india-bucket', Key='logistic/processed/logitrack_processed.csv', Body=buf2.getvalue())

    # YOUR CODE: list each prefix
    raw_list  = s3.list_objects_v2(Bucket='logitrack-india-bucket', Prefix='logistic/raw/')
    proc_list = s3.list_objects_v2(Bucket='logitrack-india-bucket', Prefix='logistic/processed/')

    print(f"Objects in logistic/raw/       : {raw_list['KeyCount']}")
    print(f"Objects in logistic/processed/ : {proc_list['KeyCount']}")

bonus_prefix_listing()

Objects in logistic/raw/       : 1
Objects in logistic/processed/ : 1


---
## Section 4 — Scoring Rubric

| Task | What is checked | Points |
|------|----------------|--------|
| T1 | Bucket `logitrack-india-bucket` created; `ap-south-1` region; HTTP 200 printed | 15 |
| T2 | `put_object` to `logistic/raw/logitrack_raw.csv`; HTTP 200 printed | 15 |
| T3 | `list_objects_v2` with `logistic/` prefix; KeyCount=1; Key + Size=38061 bytes | 20 |
| T4 | `get_object` → `pd.read_csv`; shape=(538, 9); all 9 columns printed | 20 |
| T5 | `drop_duplicates` → 530 rows; `delay_flag` applied; 153 High Delay; processed HTTP 200 | 10 |
| T6 | NRA: Number from printed output (XpressBees 11.42); Reason = structural mechanism; Action = specific + committed | 10 |
| ★ Bonus | raw/ → 1 object; processed/ → 1 object; both from separate list calls | 10★ |
| **Total** | | **90 + 10★** |

### NRA Deduction Rules (T6)
| Error type | Deduction |
|------------|-----------|
| Number not from printed output, or wrong carrier/days | −4 |
| Reason describes outcome ("delays increase") instead of mechanism ("fewer direct hub connections") | −3 |
| Action uses "may", "could", "would likely", "consider" | −2 |
| Action missing carrier name, transit threshold, or zone | −2 |

### Key Takeaway
> S3 is not a database — it's an **addressable byte store**. The pattern `BytesIO → put_object / get_object → BytesIO → pandas` is the fundamental primitive every cloud ETL uses, at any scale. Master this before touching Airflow, Glue, or Lambda.

### Interview Framing
> *"I've built ETL pipelines where raw files land in an S3 raw/ prefix, a transform step cleans and enriches them, and results write to a processed/ prefix. boto3's `put_object` / `get_object` with a BytesIO buffer is the standard pattern. I unit-test all S3 code using moto — this keeps CI fast with no real AWS dependencies. In production I replace hardcoded credentials with IAM roles or environment-injected secrets."*

### GitHub Commit
```
feat: Day185 - AWS S3 Part 1 boto3 Upload Download [score/90+10★]
```
